# QM9: GNN vs Equivariant GNN

See if encoding rotational symmetry (E(3) equivariance) improves molecular property prediction.

In [ ]:
! pip3 install -q flax optax torch-geometric

In [ ]:
import flax.linen as nn
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
from torch_geometric.datasets import QM9

## 1. Load QM9 and prepare data

Same padded-array setup as the GNN notebook.

In [ ]:
dataset = QM9(root="./data/qm9")

ELEMENT_TO_IDX = {1: 0, 6: 1, 7: 2, 8: 3, 9: 4}
N_ELEMENTS = 5
MAX_ATOMS = 29
D_HIDDEN = 64
D_MSG = 16  # smaller message MLP — the per-edge computation is the bottleneck


def load_data(n_samples=8000):
    rng = np.random.default_rng(42)
    perm = rng.permutation(len(dataset))[:n_samples]

    X_elem = np.zeros((n_samples, MAX_ATOMS, N_ELEMENTS), dtype=np.float32)
    X_pos = np.zeros((n_samples, MAX_ATOMS, 3), dtype=np.float32)
    adj = np.zeros((n_samples, MAX_ATOMS, MAX_ATOMS), dtype=np.float32)
    masks = np.zeros((n_samples, MAX_ATOMS), dtype=np.float32)
    y = np.zeros(n_samples, dtype=np.float32)

    for i, idx in enumerate(perm):
        d = dataset[int(idx)]
        z, pos = d.z.numpy(), d.pos.numpy()
        n = len(z)
        y[i] = d.y[0, 4].item()
        for j, atom in enumerate(z):
            X_elem[i, j, ELEMENT_TO_IDX[int(atom)]] = 1.0
            X_pos[i, j] = pos[j]
            masks[i, j] = 1.0
        edge_index = d.edge_index.numpy()
        for e in range(edge_index.shape[1]):
            adj[i, edge_index[0, e], edge_index[1, e]] = 1.0
        for j in range(n):
            adj[i, j, j] = 1.0

    y_mean, y_std = y.mean(), y.std()
    y = (y - y_mean) / y_std
    n_train = int(0.8 * n_samples)
    # Combined features for GNN: one-hot + xyz
    X_atoms = np.concatenate([X_elem, X_pos], axis=-1)

    print(f"Train: {n_train}, Val: {n_samples - n_train}")
    return {
        "atoms": (X_atoms[:n_train], X_atoms[n_train:]),
        "elem": (X_elem[:n_train], X_elem[n_train:]),
        "pos": (X_pos[:n_train], X_pos[n_train:]),
        "adj": (adj[:n_train], adj[n_train:]),
        "masks": (masks[:n_train], masks[n_train:]),
        "y": (y[:n_train], y[n_train:]),
        "y_std": y_std,
    }


data = load_data()

In [ ]:
# Precompute normalized adjacency for GNN
def normalize_adj(adj):
    deg = adj.sum(axis=-1, keepdims=True)
    deg_inv_sqrt = jnp.where(deg > 0, 1.0 / jnp.sqrt(deg), 0.0)
    return adj * deg_inv_sqrt * deg_inv_sqrt.transpose(0, 2, 1)


adj_norm_tr = np.array(normalize_adj(jnp.array(data["adj"][0])))
adj_norm_val = np.array(normalize_adj(jnp.array(data["adj"][1])))

## Shared training loop

In [ ]:
def train(model, apply_fn, params, train_data, val_data, y_std, n_epochs=5, batch_size=128, lr=1e-3):
    optimizer = optax.adam(lr)
    opt_state = optimizer.init(params)
    y_tr, y_val = train_data[-1], val_data[-1]

    @jax.jit
    def step(params, opt_state, *batch):
        y = batch[-1]

        def loss_fn(p):
            return jnp.mean((apply_fn(p, *batch[:-1]) - y) ** 2)

        loss, grads = jax.value_and_grad(loss_fn)(params)
        updates, new_opt = optimizer.update(grads, opt_state, params)
        return optax.apply_updates(params, updates), new_opt, loss

    train_rmses, val_rmses = [], []

    # Warmup JIT
    batch = tuple(arr[:batch_size] for arr in train_data)
    params, opt_state, _ = step(params, opt_state, *batch)

    for epoch in range(n_epochs):
        perm = np.random.permutation(len(y_tr))
        for i in range(0, len(y_tr) - batch_size + 1, batch_size):
            idx = perm[i : i + batch_size]
            batch = tuple(arr[idx] for arr in train_data)
            params, opt_state, _ = step(params, opt_state, *batch)

        va_pred = apply_fn(params, *val_data[:-1])
        va_rmse = float(jnp.sqrt(jnp.mean((va_pred - y_val) ** 2)) * y_std)
        val_rmses.append(va_rmse)

        tr_pred = apply_fn(params, *train_data[:-1])
        tr_rmse = float(jnp.sqrt(jnp.mean((tr_pred - y_tr) ** 2)) * y_std)
        train_rmses.append(tr_rmse)

        if (epoch + 1) % max(1, n_epochs // 5) == 0 or epoch == 0:
            print(f"Epoch {epoch + 1:3d} | Train: {tr_rmse:.3f} eV | Val: {va_rmse:.3f} eV")

    return params, train_rmses, val_rmses

## 2. GNN (GCN-style)

A standard GCN: aggregate neighbor features via $\tilde{A} H$, then transform. Fast (just matrix multiplies), but doesn't use 3D geometry beyond what's in the node features.

We train it for 50 epochs — it converges in seconds.

In [ ]:
class GNN(nn.Module):
    @nn.compact
    def __call__(self, x, adj, mask):
        # x: [batch, n, 8] (one-hot + xyz), adj: [batch, n, n], mask: [batch, n]
        h = adj @ x
        h = nn.relu(nn.Dense(D_HIDDEN)(h))
        h = adj @ h
        h = nn.relu(nn.Dense(D_HIDDEN)(h))
        # Mean pool
        h = h * mask[:, :, None]
        h = h.sum(axis=1) / jnp.maximum(mask.sum(axis=1, keepdims=True), 1.0)
        h = nn.relu(nn.Dense(32)(h))
        return nn.Dense(1)(h).squeeze(-1)


gnn = GNN()
gnn_params = gnn.init(
    jax.random.PRNGKey(0),
    data["atoms"][0][:1],
    adj_norm_tr[:1],
    data["masks"][0][:1],
)

gnn_params, gnn_tr, gnn_val = train(
    gnn,
    lambda p, x, a, m: gnn.apply(p, x, a, m),
    gnn_params,
    train_data=(data["atoms"][0], adj_norm_tr, data["masks"][0], data["y"][0]),
    val_data=(data["atoms"][1], adj_norm_val, data["masks"][1], data["y"][1]),
    y_std=data["y_std"],
    n_epochs=50,
)

## 3. EGNN — E(3) Equivariant GNN

Each layer updates both **scalar features** $h_i$ and **coordinates** $x_i$:

$$m_{ij} = \phi_e(h_i, h_j, \|x_i - x_j\|^2) \qquad \text{scalar messages}$$

$$x_i' = x_i + \sum_j (x_i - x_j) \cdot \phi_x(m_{ij}) \qquad \text{coordinate update}$$

$$h_i' = h_i + \phi_h(h_i, \textstyle\sum_j m_{ij}) \qquad \text{feature update}$$

Position updates use **relative vectors** scaled by **learned scalars** — so they rotate correctly under any rotation.

In [ ]:
class EGNN(nn.Module):
    @nn.compact
    def __call__(self, elem, pos, adj, mask):
        # elem: [batch, n, 5], pos: [batch, n, 3]
        # adj: [batch, n, n], mask: [batch, n]

        h = nn.Dense(D_HIDDEN)(elem)  # embed element types
        x = pos  # mutable coordinates

        for _ in range(3):
            # Pairwise relative vectors and squared distances
            rel = x[:, :, None, :] - x[:, None, :, :]  # [batch, n, n, 3]
            dist_sq = jnp.sum(rel**2, axis=-1, keepdims=True)  # [batch, n, n, 1]

            # Message: separate projections, added with broadcasting (no tiling needed)
            # Dense([h_i; h_j; d²]) ≈ Dense_1(h_i) + Dense_2(h_j) + Dense_3(d²)
            m_ij = nn.relu(
                nn.Dense(D_MSG)(h)[:, :, None, :]  # [batch, n, 1, D_MSG]
                + nn.Dense(D_MSG)(h)[:, None, :, :]  # [batch, 1, n, D_MSG]
                + nn.Dense(D_MSG)(dist_sq)  # [batch, n, n, D_MSG]
            )
            m_ij = m_ij * adj[:, :, :, None]  # zero out non-edges

            # Position update: scalar * relative vector
            x_weight = jnp.tanh(nn.Dense(1)(m_ij))  # bounded scalar per edge
            x_shift = (rel * x_weight * adj[:, :, :, None]).sum(axis=2)
            x = x + x_shift

            # Feature update with residual
            agg = m_ij.sum(axis=2)
            h = h + nn.relu(nn.Dense(D_HIDDEN)(jnp.concatenate([h, agg], axis=-1)))

        # Pool + predict
        h = h * mask[:, :, None]
        h = h.sum(axis=1) / jnp.maximum(mask.sum(axis=1, keepdims=True), 1.0)
        h = nn.relu(nn.Dense(D_HIDDEN)(h))
        return nn.Dense(1)(h).squeeze(-1)


egnn = EGNN()
egnn_params = egnn.init(
    jax.random.PRNGKey(0),
    data["elem"][0][:1],
    data["pos"][0][:1],
    data["adj"][0][:1],
    data["masks"][0][:1],
)

egnn_params, egnn_tr, egnn_val = train(
    egnn,
    lambda p, e, x, a, m: egnn.apply(p, e, x, a, m),
    egnn_params,
    train_data=(data["elem"][0], data["pos"][0], data["adj"][0], data["masks"][0], data["y"][0]),
    val_data=(data["elem"][1], data["pos"][1], data["adj"][1], data["masks"][1], data["y"][1]),
    y_std=data["y_std"],
    n_epochs=10,
)

## 4. Compare

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Learning curves
ax1.plot(
    np.arange(1, len(gnn_val) + 1), gnn_val, "-", label=f"GNN (50 ep, {gnn_val[-1]:.3f} eV)", color="C0", alpha=0.7
)
ax1.plot(np.arange(1, len(egnn_val) + 1), egnn_val, "s-", label=f"EGNN (5 ep, {egnn_val[-1]:.3f} eV)", color="C2")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Validation RMSE (eV)")
ax1.legend()
ax1.set_title("Learning curves")

# Final bar chart
names = ["GNN\n(50 epochs)", "EGNN\n(5 epochs)"]
final_vals = [gnn_val[-1], egnn_val[-1]]
ax2.bar(names, final_vals, color=["C0", "C2"])
ax2.set_ylabel("Validation RMSE (eV)")
ax2.set_title("Final performance")

plt.tight_layout()
plt.show()

print(f"\nGNN  (50 epochs): {gnn_val[-1]:.3f} eV")
print(f"EGNN  (5 epochs): {egnn_val[-1]:.3f} eV")

<![CDATA[Two kinds of symmetry, both by construction:

- **Equivariance** (cell above): the EGNN layer's coordinate outputs transform correctly under rotation — $f(Rx) = Rf(x)$. This comes from using relative vectors scaled by learned scalars.

- **Invariance** (this cell): the full model's scalar prediction is identical regardless of rotation — because the final mean-pooling over node features discards all directional information.

A standard MLP on raw coordinates would give wildly different predictions for each rotation.]]>